# 📱 S6E8: Predicting Smartphone Addiction — Complete Multi-Model & Stacking Solution
### Integrating Golden Insights, GBDT Diversity (LGBM, XGB, CatBoost), Factorization Machines, & Meta-Model Stacking

Selamat datang di notebook solusi lengkap untuk kompetisi **Kaggle Playground Series S6E8: Predicting Smartphone Addiction**.

Notebook ini mengimplementasikan seluruh **Roadmap to 0.9709+** secara end-to-end dengan struktur direktori rapi dan modular:

1. **⚡ Golden Feature Engineering:**
   - Fitur residual $\text{other\_screen} = \text{daily} - (\text{social} + \text{gaming} + \text{work\_study})$ yang mengeksploitasi *time budget constraint* generator Kaggle (univariat AUC ~0.76).
   - Fitur rasio weekend, fraksi aktivitas, rasio interaksi notifikasi/buka aplikasi, dan *missingness count*.

2. **🌲 Model Diversity 1 — Gradient Boosted Trees (LGBM, XGBoost, CatBoost):**
   - **LightGBM:** Konfigurasi cepat & akurat dengan `max_bin=512`, `learning_rate=0.05`.
   - **XGBoost:** `tree_method='hist'` dengan `max_depth=7` dan *native categorical support*.
   - **CatBoost:** `CatBoostClassifier` dengan penanganan *categorical features* tingkat lanjut.

3. **🧠 Model Diversity 2 — Factorization Machine (FM) PyTorch pada Value Lattice:**
   - Arsitektur *bilinear* low-rank interaction $\langle v_i, v_j \rangle$ untuk menangkap relasi *discrete lookup keys* antar-fitur integer dengan representasi *embedding vector*.
   - Menghasilkan prediksi yang sangat terdekorrelasi (korelasi ~0.984 terhadap pohon).

4. **🥞 Meta-Model Stacking & Regime Mixture Blending:**
   - Menggabungkan **Global Logistic Stack ($C=1.0$)** dan **Riponce's Missingness Regime Stack ($C=0.03$)** dengan bobot $2/3 \text{ Regime} + 1/3 \text{ Global}$.
   - *Percentile Rank Blending* untuk menghasilkan submission akhir berpeluang skor puncak.

## 1. Setup & Robust Path Management (Auto-detect Root & Kaggle)

In [ ]:
import os
import glob
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import logit, expit
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Model Libraries
import lightgbm as lgb

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('⚠️ xgboost belum terinstall. Install dengan: pip install xgboost')

try:
    import catboost as cb
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print('⚠️ catboost belum terinstall. Install dengan: pip install catboost')

try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print('⚠️ torch belum terinstall. Install dengan: pip install torch')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 110

# --- Robust Project Path Detection ---
def get_project_root() -> Path:
    if Path('../data/raw').exists() or Path('../data').exists():
        return Path('..').resolve()
    if Path('data/raw').exists() or Path('data').exists():
        return Path('.').resolve()
    if Path('/kaggle/working').exists():
        return Path('/kaggle/working').resolve()
    return Path('.').resolve()

PROJECT_ROOT = get_project_root()

CANDIDATES = [
    PROJECT_ROOT / 'data' / 'raw',
    PROJECT_ROOT / 'data',
    Path('/kaggle/input/playground-series-s6e8'),
    Path('/kaggle/input/competitions/playground-series-s6e8'),
]
DATA_DIR = next((p for p in CANDIDATES if (p / 'train.csv').exists()), None)

if DATA_DIR is None:
    hits = glob.glob('**/train.csv', recursive=True)
    if hits:
        DATA_DIR = Path(os.path.dirname(hits[0])).resolve()
    else:
        raise FileNotFoundError('train.csv tidak ditemukan. Pastikan dataset ada di data/raw/ atau /kaggle/input/')

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
SUBMISSION_DIR = PROJECT_ROOT / 'submissions'

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

print(f'✅ Project Root   : {PROJECT_ROOT}')
print(f'✅ Raw Data Dir   : {DATA_DIR}')
print(f'✅ Processed Dir  : {PROCESSED_DIR}')
print(f'✅ Submission Dir : {SUBMISSION_DIR}')

## 2. Data Loading & Target Distribution

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sub_df = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET = 'addicted_label'
ID_COL = 'id'

print(f'Train shape : {train_df.shape}')
print(f'Test shape  : {test_df.shape}')
print(f'Target distribution in Train:\n{train_df[TARGET].value_counts(normalize=True).round(4)}')
train_df.head()

## 3. Golden Feature Engineering

In [ ]:
RAW_NUM_COLS = [
    'age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
    'work_study_hours', 'sleep_hours', 'notifications_per_day',
    'app_opens_per_day', 'weekend_screen_time'
]
CAT_COLS = ['gender', 'stress_level', 'academic_work_impact']

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    
    dst = data['daily_screen_time_hours']
    sm = data['social_media_hours']
    gm = data['gaming_hours']
    ws = data['work_study_hours']
    wst = data['weekend_screen_time']
    notif = data['notifications_per_day']
    opens = data['app_opens_per_day']
    
    # 1. Residual Budget Feature (Golden Discovery)
    sgw = sm + gm + ws
    other_screen = dst - sgw
    
    data['sgw_sum'] = sgw
    data['other_screen'] = other_screen
    data['other_frac'] = (other_screen / (dst.clip(lower=0.1) + 0.01))
    
    # 2. Weekend dynamics
    data['weekend_ratio'] = wst / (dst + 0.01)
    data['weekend_diff'] = wst - dst
    data['weekend_minus_other'] = wst - other_screen
    
    # 3. Activity fractions
    data['social_ratio'] = sm / (dst + 0.01)
    data['gaming_ratio'] = gm / (dst + 0.01)
    data['work_ratio'] = ws / (dst + 0.01)
    
    # 4. App interaction ratios
    data['notif_per_open'] = notif / (opens + 1.0)
    data['opens_per_screen_hr'] = opens / (dst + 0.1)
    data['notif_per_screen_hr'] = notif / (dst + 0.1)
    
    # 5. Missingness pattern count
    data['miss_count'] = data[RAW_NUM_COLS + CAT_COLS].isna().sum(axis=1)
    
    # Convert categoricals to category dtype
    for col in CAT_COLS:
        data[col] = data[col].astype('category')
        
    return data

print('Engineering features on train & test sets...')
X_train_full = engineer_features(train_df)
X_test_full = engineer_features(test_df)

y_train = train_df[TARGET].values

FEATURE_COLS = [c for c in X_train_full.columns if c not in [ID_COL, TARGET]]
print(f'✅ Total features: {len(FEATURE_COLS)}')
print(f'Feature list: {FEATURE_COLS}')

# Standard 5-Fold Stratified Split (Frozen Seed 42 for all models)
N_SPLITS = 5
SEED = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(X_train_full, y_train))

# Global dictionary to store OOF and Test predictions from all model families
oof_dict = {}
test_dict = {}

## 4. Model 1: LightGBM (5-Fold Stratified CV)

Model pertama berbasis **LightGBM** dengan `max_bin=512`, `learning_rate=0.05`, dan evaluasi cepat pada `dval`.

In [ ]:
print('--- Training Model 1: LightGBM ---')

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_bin': 512,
    'min_child_samples': 100,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': SEED,
    'verbose': -1,
    'n_jobs': -1
}

oof_lgb = np.zeros(len(X_train_full))
test_lgb = np.zeros(len(X_test_full))
lgb_importances = np.zeros(len(FEATURE_COLS))

t0 = time.time()
for fold, (train_idx, val_idx) in enumerate(folds):
    X_tr, y_tr = X_train_full.iloc[train_idx][FEATURE_COLS], y_train[train_idx]
    X_va, y_va = X_train_full.iloc[val_idx][FEATURE_COLS], y_train[val_idx]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=CAT_COLS, free_raw_data=False)
    dval = lgb.Dataset(X_va, label=y_va, categorical_feature=CAT_COLS, reference=dtrain, free_raw_data=False)
    
    model = lgb.train(
        params=lgb_params,
        train_set=dtrain,
        num_boost_round=1500,
        valid_sets=[dval],
        valid_names=['valid'],
        callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(0)]
    )
    
    val_pred = model.predict(X_va, num_iteration=model.best_iteration)
    oof_lgb[val_idx] = val_pred
    test_lgb += model.predict(X_test_full[FEATURE_COLS], num_iteration=model.best_iteration) / N_SPLITS
    lgb_importances += model.feature_importance(importance_type='gain') / N_SPLITS
    
    fold_auc = roc_auc_score(y_va, val_pred)
    print(f'  Fold {fold + 1}/{N_SPLITS} AUC: {fold_auc:.5f} (Best Iter: {model.best_iteration})')

auc_lgb = roc_auc_score(y_train, oof_lgb)
oof_dict['lgbm'] = oof_lgb
test_dict['lgbm'] = test_lgb
print(f'✅ LightGBM OOF ROC-AUC: {auc_lgb:.5f} (Elapsed: {(time.time() - t0)/60:.2f} min)')

## 5. Model 2: XGBoost (5-Fold Stratified CV)

Model kedua menggunakan **XGBoost** dengan `tree_method='hist'`, `enable_categorical=True`, dan `max_depth=7`.

In [ ]:
if HAS_XGB:
    print('--- Training Model 2: XGBoost ---')
    
    xgb_params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'learning_rate': 0.05,
        'max_depth': 7,
        'max_bin': 512,
        'subsample': 0.8,
        'colsample_bytree': 0.7,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': SEED,
        'enable_categorical': True,
        'n_jobs': -1
    }
    
    oof_xgb = np.zeros(len(X_train_full))
    test_xgb = np.zeros(len(X_test_full))
    
    t0 = time.time()
    for fold, (train_idx, val_idx) in enumerate(folds):
        X_tr, y_tr = X_train_full.iloc[train_idx][FEATURE_COLS], y_train[train_idx]
        X_va, y_va = X_train_full.iloc[val_idx][FEATURE_COLS], y_train[val_idx]
        
        dtr = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
        dva = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
        dte = xgb.DMatrix(X_test_full[FEATURE_COLS], enable_categorical=True)
        
        model_xgb = xgb.train(
            params=xgb_params,
            dtrain=dtr,
            num_boost_round=1500,
            evals=[(dva, 'valid')],
            early_stopping_rounds=80,
            verbose_eval=False
        )
        
        val_pred = model_xgb.predict(dva)
        oof_xgb[val_idx] = val_pred
        test_xgb += model_xgb.predict(dte) / N_SPLITS
        
        fold_auc = roc_auc_score(y_va, val_pred)
        print(f'  Fold {fold + 1}/{N_SPLITS} AUC: {fold_auc:.5f} (Best Iter: {model_xgb.best_iteration})')
        
    auc_xgb = roc_auc_score(y_train, oof_xgb)
    oof_dict['xgboost'] = oof_xgb
    test_dict['xgboost'] = test_xgb
    print(f'✅ XGBoost OOF ROC-AUC: {auc_xgb:.5f} (Elapsed: {(time.time() - t0)/60:.2f} min)')
else:
    print('Skipping XGBoost (library not installed).')

## 6. Model 3: CatBoost (5-Fold Stratified CV)

Model ketiga menggunakan **CatBoost** dengan algoritma *symmetric trees* yang sangat tangguh terhadap overfitting.

In [ ]:
if HAS_CATBOOST:
    print('--- Training Model 3: CatBoost ---')
    
    oof_cb = np.zeros(len(X_train_full))
    test_cb = np.zeros(len(X_test_full))
    
    cb_cat_cols = CAT_COLS
    X_train_cb = X_train_full[FEATURE_COLS].copy()
    X_test_cb = X_test_full[FEATURE_COLS].copy()
    for c in cb_cat_cols:
        X_train_cb[c] = X_train_cb[c].astype(str).fillna('missing')
        X_test_cb[c] = X_test_cb[c].astype(str).fillna('missing')
        
    t0 = time.time()
    for fold, (train_idx, val_idx) in enumerate(folds):
        X_tr, y_tr = X_train_cb.iloc[train_idx], y_train[train_idx]
        X_va, y_va = X_train_cb.iloc[val_idx], y_train[val_idx]
        
        cb_model = cb.CatBoostClassifier(
            loss_function='Logloss',
            eval_metric='AUC',
            learning_rate=0.06,
            depth=6,
            iterations=1200,
            cat_features=cb_cat_cols,
            random_seed=SEED,
            verbose=0,
            thread_count=-1
        )
        
        cb_model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            early_stopping_rounds=80,
            verbose=False
        )
        
        val_pred = cb_model.predict_proba(X_va)[:, 1]
        oof_cb[val_idx] = val_pred
        test_cb += cb_model.predict_proba(X_test_cb)[:, 1] / N_SPLITS
        
        fold_auc = roc_auc_score(y_va, val_pred)
        print(f'  Fold {fold + 1}/{N_SPLITS} AUC: {fold_auc:.5f} (Best Iter: {cb_model.get_best_iteration()})')
        
    auc_cb = roc_auc_score(y_train, oof_cb)
    oof_dict['catboost'] = oof_cb
    test_dict['catboost'] = test_cb
    print(f'✅ CatBoost OOF ROC-AUC: {auc_cb:.5f} (Elapsed: {(time.time() - t0)/60:.2f} min)')
else:
    print('Skipping CatBoost (library not installed).')

## 7. Model 4: PyTorch Factorization Machine (FM) pada Value Lattice

Arsitektur *Deep Learning Tabular / Factorization Machine* bilinear:
$$\hat{y} = b + \sum_f w_{f, x_f} + \sum_{f < g} \langle v_{f, x_f},\, v_{g, x_g} \rangle + \text{DeepHead}(e)$$

Model ini mempelajari representasi interaksi antar-nilai integer diskrit secara bersamaan (*joint estimation*), memberikan diversitas paling berharga ke dalam ensemble.

In [ ]:
if HAS_TORCH:
    print('--- Training Model 4: PyTorch Factorization Machine (FM) ---')
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'PyTorch Device: {device}')
    
    class FactorizationMachine(nn.Module):
        def __init__(self, total_vocab, n_fields, k=16, deep=128, drop=0.1):
            super().__init__()
            self.v = nn.Embedding(total_vocab, k)      # Lattice interaction vectors
            self.w = nn.Embedding(total_vocab, 1)      # Linear weights per value
            nn.init.normal_(self.v.weight, std=0.01)
            nn.init.zeros_(self.w.weight)
            self.bias = nn.Parameter(torch.zeros(1))
            
            self.deep = nn.Sequential(
                nn.Linear(n_fields * k, deep),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(deep, deep // 2),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(deep // 2, 1),
            ) if deep else None
            
        def forward(self, idx):
            e = self.v(idx)                            # (batch, fields, k)
            s = e.sum(dim=1)                           # (batch, k)
            # Bilinear interaction O(k * fields)
            interaction = 0.5 * (s.pow(2).sum(dim=1) - e.pow(2).sum(dim=(1, 2)))
            out = self.bias + self.w(idx).sum(dim=(1, 2)) + interaction
            if self.deep is not None:
                out = out + self.deep(e.flatten(1)).squeeze(-1)
            return out
            
    # Build field integer codes with rare bucketing
    both_df = pd.concat([train_df[RAW_NUM_COLS + CAT_COLS], test_df[RAW_NUM_COLS + CAT_COLS]], ignore_index=True)
    
    def build_fm_fields(df, min_count=15):
        codes, vocabs, names = [], [], []
        for c in df.columns:
            s = df[c].astype(str).where(df[c].notna(), None)
            counts = s.value_counts()
            keep = counts[counts >= min_count].index
            mapping = {v: i + 2 for i, v in enumerate(keep)}   # 0 = NaN, 1 = rare
            code = s.map(mapping)
            code = code.where(s.isna() | code.notna(), 1.0)
            codes.append(code.fillna(0).to_numpy(np.int64))
            vocabs.append(len(keep) + 2)
            names.append(c)
        return np.stack(codes, 1), np.asarray(vocabs), names
        
    fm_codes, fm_vocabs, fm_names = build_fm_fields(both_df)
    fm_offsets = np.concatenate([[0], np.cumsum(fm_vocabs)[:-1]])
    FM_IDX = torch.from_numpy(fm_codes + fm_offsets[None, :])
    TOTAL_VOCAB = int(fm_vocabs.sum())
    N_FIELDS = len(fm_names)
    
    print(f'FM Lattice: {N_FIELDS} fields, {TOTAL_VOCAB:,} embedding vocabulary rows')
    
    # Fast training PyTorch loop
    oof_fm = np.zeros(len(train_df))
    test_fm = np.zeros(len(test_df))
    
    I_train = FM_IDX[:len(train_df)]
    I_test = FM_IDX[len(train_df):].to(device)
    Y_train_t = torch.from_numpy(y_train.astype(np.float32))
    
    BATCH_SIZE = 4096
    EPOCHS = 12
    
    t0 = time.time()
    for fold, (train_idx, val_idx) in enumerate(folds):
        model_fm = FactorizationMachine(TOTAL_VOCAB, N_FIELDS, k=16, deep=128).to(device)
        optimizer = torch.optim.AdamW(model_fm.parameters(), lr=0.005, weight_decay=1e-4)
        criterion = nn.BCEWithLogitsLoss()
        
        I_tr, Y_tr = I_train[train_idx].to(device), Y_train_t[train_idx].to(device)
        I_va, Y_va = I_train[val_idx].to(device), Y_train_t[val_idx]
        
        best_auc = 0.0
        best_val_pred = None
        
        for ep in range(EPOCHS):
            model_fm.train()
            perm = torch.randperm(len(train_idx), device=device)
            for i in range(0, len(train_idx), BATCH_SIZE):
                sl = perm[i:i + BATCH_SIZE]
                optimizer.zero_grad()
                out = model_fm(I_tr[sl])
                loss = criterion(out, Y_tr[sl])
                loss.backward()
                optimizer.step()
                
            model_fm.eval()
            with torch.no_grad():
                val_logits = torch.cat([model_fm(I_va[j:j + 32768]) for j in range(0, len(val_idx), 32768)]).cpu().numpy()
                val_probs = expit(val_logits)
                ep_auc = roc_auc_score(Y_va, val_probs)
                if ep_auc > best_auc:
                    best_auc = ep_auc
                    best_val_pred = val_probs
                    
        oof_fm[val_idx] = best_val_pred
        
        with torch.no_grad():
            test_logits = torch.cat([model_fm(I_test[j:j + 32768]) for j in range(0, len(test_df), 32768)]).cpu().numpy()
            test_fm += expit(test_logits) / N_SPLITS
            
        print(f'  Fold {fold + 1}/{N_SPLITS} Best AUC: {best_auc:.5f}')
        
    auc_fm = roc_auc_score(y_train, oof_fm)
    oof_dict['fm'] = oof_fm
    test_dict['fm'] = test_fm
    print(f'✅ Factorization Machine OOF ROC-AUC: {auc_fm:.5f} (Elapsed: {(time.time() - t0)/60:.2f} min)')
else:
    print('Skipping PyTorch FM (torch not installed).')

## 8. Model Correlation & Diversity Inspection

Kunci utama keberhasilan *stacking* adalah **korelasi antar-model yang cukup rendah (< 0.995)** sehingga model saling melengkapi.

In [ ]:
oof_df_all = pd.DataFrame(oof_dict)

print('--- Model Summary & Individual OOF AUC Scores ---')
for name in oof_dict:
    print(f'  {name:<12s} : OOF ROC-AUC = {roc_auc_score(y_train, oof_dict[name]):.5f}')
    
print('\n--- Spearman Rank Correlation Matrix Between Models ---')
corr_matrix = oof_df_all.corr(method='spearman')
display(corr_matrix.round(4))

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap='Blues', vmin=0.97, vmax=1.0, fmt='.4f')
plt.title('Spearman Rank Correlation Across Model Families')
plt.tight_layout()
plt.show()

## 9. Meta-Model Stacking (Global vs Missingness-Regime Logistic Mixture)

Berdasarkan temuan di Notebook master publik:
1. **Global Logistic Stack ($C=1.0$):** Menggabungkan logit dari seluruh model.
2. **Regime Stack ($C=0.03$):** Menambahkan interaksi *is_complete*, *severe_missing ($\ge 4$)*, dan *disagreement std*.
3. **Percentile Rank Mixture:** Mengambil rata-rata rank dari $\frac{2}{3} \text{ Regime} + \frac{1}{3} \text{ Global}$.

In [ ]:
EPS = 1e-6

# 1. Convert probabilities to logits for linear meta-modeling
model_names = list(oof_dict.keys())
O_logits = np.column_stack([logit(np.clip(oof_dict[m], EPS, 1 - EPS)) for m in model_names])
T_logits = np.column_stack([logit(np.clip(test_dict[m], EPS, 1 - EPS)) for m in model_names])

miss_train = X_train_full['miss_count'].values
miss_test = X_test_full['miss_count'].values

# 2. Regime Design Matrix Function
def build_regime_matrix(lg, missing):
    complete = (missing == 0).astype(np.float64)[:, None]
    severe = (missing >= 4).astype(np.float64)[:, None]
    d = lg.std(axis=1, keepdims=True)
    dn = (d - d.mean()) / (d.std() + 1e-6)
    agg = np.column_stack([lg.mean(1), lg.std(1), lg.max(1) - lg.min(1), complete[:, 0], severe[:, 0]])
    return np.column_stack([lg, lg * complete, lg * severe, lg * dn, agg])

MR_train = build_regime_matrix(O_logits, miss_train)
MR_test = build_regime_matrix(T_logits, miss_test)

# 3. Out-Of-Fold Honest Meta-Model Training Function
def train_honest_meta(X_meta, C_val):
    meta_oof = np.zeros(len(y_train))
    for train_idx, val_idx in folds:
        scaler = StandardScaler().fit(X_meta[train_idx])
        clf = LogisticRegression(C=C_val, max_iter=2000, solver='lbfgs')
        clf.fit(scaler.transform(X_meta[train_idx]), y_train[train_idx])
        meta_oof[val_idx] = clf.predict_proba(scaler.transform(X_meta[val_idx]))[:, 1]
    return meta_oof

oof_meta_global = train_honest_meta(O_logits, C_val=1.0)
oof_meta_regime = train_honest_meta(MR_train, C_val=0.03)

# 4. Percentile Rank Mixture Function
def to_pct(arr):
    return rankdata(arr) / len(arr)

rank_global = to_pct(oof_meta_global)
rank_regime = to_pct(oof_meta_regime)

# 2/3 Regime + 1/3 Global mixture
oof_mixture = (2.0 / 3.0) * rank_regime + (1.0 / 3.0) * rank_global

print('--- Meta-Model Validation Results ---')
print(f'  Global Logistic Stack (C=1.0)  OOF AUC : {roc_auc_score(y_train, oof_meta_global):.5f}')
print(f'  Regime Logistic Stack (C=0.03) OOF AUC : {roc_auc_score(y_train, oof_meta_regime):.5f}')
print(f'  🏆 Mixture Meta-Stack (2/3 + 1/3) OOF AUC : {roc_auc_score(y_train, oof_mixture):.5f}')

# Simple Mean Blend Comparison
oof_simple_blend = np.mean(list(oof_dict.values()), axis=0)
print(f'  Simple Average Blend            OOF AUC : {roc_auc_score(y_train, oof_simple_blend):.5f}')

## 10. Generating Final Test Predictions & Sanity Checks

Melatih meta-model pada seluruh training set untuk menghasilkan file submission akhir.

In [ ]:
# Fit Global Meta on all train
sc_g = StandardScaler().fit(O_logits)
meta_g = LogisticRegression(C=1.0, max_iter=2000, solver='lbfgs').fit(sc_g.transform(O_logits), y_train)
test_meta_global = meta_g.predict_proba(sc_g.transform(T_logits))[:, 1]

# Fit Regime Meta on all train
sc_r = StandardScaler().fit(MR_train)
meta_r = LogisticRegression(C=0.03, max_iter=2000, solver='lbfgs').fit(sc_r.transform(MR_train), y_train)
test_meta_regime = meta_r.predict_proba(sc_r.transform(MR_test))[:, 1]

# Final Rank Blended Prediction
final_test_pred = (2.0 / 3.0) * to_pct(test_meta_regime) + (1.0 / 3.0) * to_pct(test_meta_global)

# Save All OOFs to CSV
oof_export_df = pd.DataFrame(oof_dict)
oof_export_df['meta_mixture'] = oof_mixture
oof_export_df[ID_COL] = train_df[ID_COL]
oof_export_df[TARGET] = y_train

oof_export_path = PROCESSED_DIR / 'oof_all_models.csv'
oof_export_df.to_csv(oof_export_path, index=False)
print(f'✅ All model OOFs saved to: {oof_export_path}')

# Final Submission DataFrame
final_submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: final_test_pred
})

# Rigorous Sanity Checks
assert len(final_submission) == len(test_df), 'Row count mismatch in submission!'
assert not final_submission[TARGET].isna().any(), 'Nulls detected in submission!'
assert final_submission[TARGET].between(0, 1).all(), 'Probabilities outside [0, 1] range!'
assert final_submission[TARGET].nunique() > 250000, 'Not enough unique ranking values!'

final_sub_path = SUBMISSION_DIR / 'submission_ensemble_top.csv'
root_sub_path = PROJECT_ROOT / 'submission.csv'

final_submission.to_csv(final_sub_path, index=False)
final_submission.to_csv(root_sub_path, index=False)

print(f'🚀 FINAL ENSEMBLE SUBMISSION CREATED SUCCESSFULLY!')
print(f'   Saved to: {final_sub_path}')
print(f'   Saved to: {root_sub_path}')
print(f'   Total Rows: {len(final_submission):,}')
print(f'   Distinct Values: {final_submission[TARGET].nunique():,}')
print('\nSubmission Head:')
display(final_submission.head(10))

## 11. Final ROC Curves & Calibration Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. ROC Curves Comparison
for name in oof_dict:
    fpr, tpr, _ = roc_curve(y_train, oof_dict[name])
    axes[0].plot(fpr, tpr, lw=1.5, label=f'{name} (AUC={roc_auc_score(y_train, oof_dict[name]):.5f})')
    
fpr_m, tpr_m, _ = roc_curve(y_train, oof_mixture)
axes[0].plot(fpr_m, tpr_m, color='red', lw=2.5, linestyle='--', label=f'Meta Mixture (AUC={roc_auc_score(y_train, oof_mixture):.5f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle=':', lw=1)
axes[0].set_title('ROC Curves Comparison Across Models')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')
axes[0].grid(True)

# 2. Final Ensemble Prediction Distribution
axes[1].hist(final_submission[TARGET], bins=60, color='#2a78d6', alpha=0.75, edgecolor='black', linewidth=0.5)
axes[1].set_title('Final Test Percentile Rank Prediction Distribution')
axes[1].set_xlabel('Rank Percentile')
axes[1].set_ylabel('Frequency')
axes[1].grid(True)

plt.tight_layout()
plt.show()